In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

MY_ID = "arzoo"
VOL = f"/Volumes/workspace/capstone_{MY_ID}/raw"
S   = f"workspace.capstone_{MY_ID}"

s     = spark.table(f"{S}.silver_complaint")
wards = spark.table(f"{S}.bronze_wards").select("ward_id", "zone")
cats  = s.select("category").distinct()
weeks = spark.range(26).select(F.expr("date_add(date'2024-12-30', cast(id*7 as int))").alias("week_start"))
spine = weeks.crossJoin(wards).crossJoin(cats)          # 26 x 40 x 8 = 8,320

k = ["week_start", "ward_id", "category"]

r = (s.withColumn("week_start", F.expr("cast(date_trunc('week', raised_ts) as date)"))
      .groupBy(*k)
      .agg(F.count("*").alias("raised"),
           F.sum(F.expr("cast(not (status='CLOSED' and not closure_known) as int)")).alias("arr_open"),
           F.sum(F.expr("cast(coalesce(sla_breach, false) as int)")).alias("breached"),
           F.count("sla_breach").alias("bn")))

c = (s.filter("closure_known")
      .withColumn("week_start", F.expr("cast(date_trunc('week', closed_ts) as date)"))
      .groupBy(*k)
      .agg(F.count("*").alias("closed"),
           F.expr("percentile(resolution_hours, 0.5)").alias("median_resolution_hours"),
           F.expr("percentile(resolution_hours, 0.9)").alias("p90_resolution_hours")))

wn = Window.partitionBy("ward_id", "category").orderBy("week_start") \
           .rowsBetween(Window.unboundedPreceding, 0)

g = (spine.join(r, k, "left").join(c, k, "left")
      .na.fill(0, ["raised", "arr_open", "closed", "breached", "bn"])
      .withColumn("open_at_week_end", F.sum(F.col("arr_open") - F.col("closed")).over(wn))
      .withColumn("breach_rate", F.expr("case when bn > 0 then breached / bn end"))
      .select("week_start", "ward_id", "zone", "category", "raised", "closed",
              "open_at_week_end", "median_resolution_hours", "p90_resolution_hours",
              "breached", "breach_rate"))

g.write.mode("overwrite").saveAsTable(f"{S}.gold_grievance_weekly")

In [0]:
g = spark.table(f"{S}.gold_grievance_weekly")
print("gold rows:", g.count())                                                  # 8,320
print("last-week backlog:", g.filter("week_start = date'2025-06-23'")
      .agg(F.sum("open_at_week_end")).first()[0])
print("Silver OPEN rows:", s.filter("status='OPEN'").count())                   # equals the line above
print("brief's formula (Silver - closure_known):", s.count() - s.filter("closure_known").count())  # above + 1,200
print("zero/negative medians:", g.filter("median_resolution_hours <= 0").count())  # 0

gold rows: 8320
last-week backlog: 8875
Silver OPEN rows: 8875
brief's formula (Silver - closure_known): 10075
zero/negative medians: 0
